### **Notebook 2 (etapa 3 y 4): Entrenamiento y evaluación de Random Forest**

In [ ]:
import pandas as pd  # Permite el manejo y análisis de estructuras de datos (DataFrames)
import numpy as np  # Facilita la realización de cálculos numéricos y manejo de matrices/arrays
import os  # Interacción con el sistema operativo (creación y verificación de rutas/directorios)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM
import time  # Medición de los tiempos de ejecución de las tareas
import joblib  # Serialización y guardado de los modelos entrenados de Machine Learning en disco
from sklearn.model_selection import GridSearchCV  # Realiza búsquedas exhaustivas de hiperparámetros mediante validación cruzada
from sklearn.model_selection import StratifiedKFold  # Divide los datos en pliegues preservando la proporción original de cada clase
from sklearn.ensemble import RandomForestClassifier  # Algoritmo principal de ensamble (Bosque Aleatorio) para clasificación
from sklearn.base import clone  # Permite clonar estimadores sin copiar los datos con los que fueron ajustados
from sklearn.preprocessing import label_binarize  # Convierte etiquetas multiclase en un formato binario (One-vs-Rest)
from sklearn.metrics import (  # Colección de funciones para evaluar el rendimiento del modelo
    f1_score, 
    average_precision_score, 
    roc_auc_score, 
    brier_score_loss, 
    classification_report 
)

def entrenar_evaluar_rf(target_name):
    """
    Descripción:
        Entrena, optimiza mediante GridSearch, y evalúa un modelo de Random Forest (Bosque Aleatorio).
        El flujo incluye el balanceo de las clases en entrenamiento, la búsqueda de hiperparámetros
        estables en el tiempo, la evaluación del modelo en el conjunto de prueba (métricas y lift), 
        la extracción de la importancia de las variables (Feature Importances) y el guardado de 
        todos los resultados y el modelo serializado (.pkl) en disco.

    Entradas:
        - target_name (str): Nombre exacto de la columna que representa la variable objetivo (target) a predecir.

    Salidas:
        - None: La función no retorna elementos directamente en memoria, pero guarda en disco local:
            1. Un archivo CSV con los resultados de la validación cruzada (GridSearch).
            2. El modelo serializado (.pkl) con la mejor configuración de hiperparámetros.
            3. Un archivo CSV con el reporte de métricas desglosado por clases.
            4. Un archivo CSV con la importancia relativa de las variables (Feature Importances).
    """
    # 1. Configuración inicial
    # Definir el directorio de lectura de datos
    dir_datos = "../../Datos/Datasets Finales" 
    # Definir y crear el directorio para almacenar los resultados del Random Forest si no existe
    dir_resultados = "../../Resultados/Resultados (etapa 3 y 4)/Random_Forest" 
    os.makedirs(dir_resultados, exist_ok=True) 

    # Lista de variables que no deben ser incluidas como características (features) predictoras
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER'] 

    print("="*60) 
    print(f"Iniciando entrenamiento y evaluación de Random Forest para la variable objetivo: {target_name.upper()}") 
    print("="*60) 

    # 2. Cargar datos de entrenamiento
    print("[1/5] Cargando datasets de entrenamiento...") 
    # Lectura del dataset con casos positivos/oncológicos
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False) 
    # Lectura del dataset con casos negativos/de control
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control.csv"), low_memory=False) 

    # 3. Crear Dataset Maestro Balanceado 
    print("[2/5] Generando muestra balanceada...") 
    # Obtener el número de registros oncológicos para equilibrar las clases
    n_onco = len(df_onco_train) 
    # Tomar una muestra aleatoria (con semilla fija) de casos de control equivalente al tamaño oncológico
    df_control_sample = df_control_train.sample(n=n_onco, random_state=42) 
    # Unificar los subconjuntos para crear el dataframe de entrenamiento final
    df_train_maestro = pd.concat([df_onco_train, df_control_sample], ignore_index=True) 

    # Eliminar dataframes intermedios y llamar al colector de basura para liberar memoria RAM
    del df_onco_train, df_control_train, df_control_sample 
    gc.collect() 

    # Filtrar las columnas para dejar únicamente las características predictoras
    features = [col for col in df_train_maestro.columns if col not in cols_excluir] 
    # Asignar features a X_train y el target a y_train
    X_train = df_train_maestro[features] 
    y_train = df_train_maestro[target_name] 
    
    # Identificar la cantidad de clases presentes para saber si es clasificación binaria o multiclase
    clases_unicas = np.unique(y_train) 
    is_multiclass = len(clases_unicas) > 2 
    
    print(f"      -> Dimensiones: {X_train.shape} | Clases: {len(clases_unicas)} (Multiclase: {is_multiclass})") 

    # 4. Configurar Grid Search (5 Pliegues) 
    print("[3/5] Configurando Grid Search CV (K=5) ...") 
    
    # Establecer la validación cruzada estratificada para preservar la proporción de clases en 5 cortes
    cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Inicializar el estimador base de Random Forest con pesos balanceados y semilla fija
    rf_base = RandomForestClassifier( 
        class_weight='balanced', 
        random_state=42, 
        n_jobs=-1  # Utilizar todos los núcleos disponibles de la CPU para procesar los árboles en paralelo
    )

    # Definir la grilla de hiperparámetros a explorar para el Random Forest
    espacio_hiperparametros = { 
        'n_estimators': [100, 200, 500],  # Cantidad de árboles en el bosque
        'max_depth': [15, 25, 35], # Controla la profundidad máxima
        'min_samples_split': [10, 20, 50]  # Obligamos a que las hojas agrupen pacientes (evita sobreajuste)
    }

    # Configurar el buscador exhaustivo optimizando el 'f1_macro'
    grid_search = GridSearchCV( 
        estimator=rf_base, 
        param_grid=espacio_hiperparametros, 
        cv=cv_estrategia, 
        scoring='f1_macro', 
        n_jobs=1,  # n_jobs=1 aquí porque el estimador base ya usa -1 (evita saturar la CPU)
        verbose=3 
    )

    # 5. Entrenar y evaluar configuraciones
    print("[4/5] Entrenando modelo y evaluando configuraciones ...") 
    # Marcar el tiempo de inicio
    inicio = time.time() 
    # Ejecutar la búsqueda de parámetros sobre los datos de entrenamiento
    grid_search.fit(X_train, y_train) 
    # Marcar tiempo de fin y calcular duración
    fin = time.time() 
    print(f"      -> Búsqueda completada en {round((fin - inicio)/60, 2)} minutos.") 

    # Transformar el resumen del GridSearch a DataFrame y exportar
    cv_resultados = pd.DataFrame(grid_search.cv_results_) 
    ruta_cv = os.path.join(dir_resultados, f"Resultados_GridSearch_RF_{target_name}.csv") 
    cv_resultados.to_csv(ruta_cv, index=False) 
    print(f"      -> Evidencia de hiperparámetros guardada en: {ruta_cv}") 
    
    # Filtrar solo las configuraciones que tienen una desviación estándar pequeña en los pliegues (estables)
    config_estables = cv_resultados[cv_resultados['std_test_score'] <= 0.10] 

    if config_estables.empty: 
        print("      ADVERTENCIA: Todas las configuraciones tienen desviación estándar (std) > 0.10.") 
        print("      Se utilizará la de mayor promedio por defecto de Scikit-Learn.") 
        # Si no hay estables, usar el modelo ganador por defecto
        mejor_modelo = grid_search.best_estimator_ 
    else: 
        # Obtener el índice de la configuración con mayor F1 Score promedio entre los resultados estables
        mejor_indice = config_estables['mean_test_score'].idxmax() 
        mejores_params = config_estables.loc[mejor_indice, 'params'] 
        mejor_f1 = config_estables.loc[mejor_indice, 'mean_test_score'] 
        mejor_std = config_estables.loc[mejor_indice, 'std_test_score'] 
        
        print(f"      -> Mejor configuración estable encontrada:") 
        print(f"         Hiperparámetros: {mejores_params}") 
        print(f"         F1-Macro Promedio: {mejor_f1:.4f} (std: {mejor_std:.4f})") 

        # Clonar el modelo base, aplicar los mejores hiperparámetros estables, y entrenar con toda la data
        mejor_modelo = clone(grid_search.estimator) 
        mejor_modelo.set_params(**mejores_params) 
        mejor_modelo.fit(X_train, y_train) 
        
    # --- GUARDADO DEL MODELO MAESTRO EN DISCO ---
    # Configurar ruta del archivo pickle para el modelo Random Forest final
    ruta_modelo = os.path.join(dir_resultados, f"Modelo_Optimo_RF_{target_name}.pkl")
    # Exportar el modelo entrenado usando joblib
    joblib.dump(mejor_modelo, ruta_modelo)
    print(f"      -> Modelo óptimo guardado en: {ruta_modelo}")
    # --------------------------------------------

    # --- MÉTRICAS DE ENTRENAMIENTO ---
    print("\n--- Rendimiento en entrenamiento: ---")
    # Realizar predicciones sobre el conjunto con el que se acaba de entrenar
    y_pred_train = mejor_modelo.predict(X_train)
    # Medir e imprimir métricas sobre la fase de entrenamiento
    if is_multiclass:
        print(f"F1-Score (Macro) Train: {f1_score(y_train, y_pred_train, average='macro'):.4f}")
    else:
        print(f"F1-Score (Clase 1) Train: {f1_score(y_train, y_pred_train, pos_label=1):.4f}")
    # ---------------------------------

    # Liberar memoria de los datos de entrenamiento
    del df_train_maestro, X_train, y_train 
    gc.collect() 

    # 6. Evaluación en el Conjunto de Prueba
    print("[5/5] Evaluando en conjunto de prueba...") 
    # Cargar los datasets de test que no fueron vistos durante la etapa de entrenamiento
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False) 
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False) 

    # Unir ambas tablas y separar features/target para evaluación
    df_test_maestro = pd.concat([df_onco_test, df_control_test], ignore_index=True) 
    X_test = df_test_maestro[features] 
    y_test = df_test_maestro[target_name] 
    total_instancias = len(y_test) 

    # Calcular las predicciones duras y las probabilidades para el set de prueba
    y_pred = mejor_modelo.predict(X_test) 
    y_pred_proba = mejor_modelo.predict_proba(X_test) 

    print("\n--- Resultados finales en evaluación ---") 
    # Mostrar por consola el reporte general de clasificación
    print(classification_report(y_test, y_pred)) 
    
    # --- EXPORTAR REPORTE A CSV ---
    # Transformar el reporte de sklearn en diccionario y luego en dataframe para exportar
    reporte_dic = classification_report(y_test, y_pred, output_dict=True)
    df_reporte = pd.DataFrame(reporte_dic).transpose()
    ruta_reporte = os.path.join(dir_resultados, f"Reporte_Desglose_Clases_RF_{target_name}.csv")
    df_reporte.to_csv(ruta_reporte)
    print(f"      -> Reporte de desglose por clases guardado en: {ruta_reporte}")
    # ------------------------------
    
    # Obtener métrica general F1-Macro
    f1_macro_val = f1_score(y_test, y_pred, average='macro') 
    
    # Evaluar métricas específicas según la naturaleza de la clasificación (Multiclase o Binaria)
    if is_multiclass: 
        # Binarizar el conjunto de prueba para que funcionen métricas especializadas One-vs-Rest (OvR)
        y_test_bin = label_binarize(y_test, classes=clases_unicas) 
        auc_roc_val = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted') 
        auprc_val = average_precision_score(y_test_bin, y_pred_proba, average='weighted') 
        
        # Calcular la métrica Brier score promediando los errores de probabilidad por cada clase
        brier_val = np.mean([brier_score_loss(y_test_bin[:, k], y_pred_proba[:, k]) for k in range(len(clases_unicas))]) 
        
        # Calcular tasa base ponderada analizando las frecuencias reales de las clases
        clases_temp, soportes_clases = np.unique(y_test, return_counts=True)
        prevalencias = [soporte / total_instancias for soporte in soportes_clases]
        tasa_base = sum([p**2 for p in prevalencias])

        # Imprimir resultados del rendimiento multiclase
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") 
        print(f"AUPRC (OvR Weighted): {auprc_val:.4f}") 
        print(f"AUC-ROC (OvR Weighted): {auc_roc_val:.4f}") 
        print(f"Brier Score (Multiclase): {brier_val:.4f}") 
            
    else: 
        # Calcular métricas para una clasificación binaria clásica
        f1_clase1_val = f1_score(y_test, y_pred, pos_label=1) 
        auc_roc_val = roc_auc_score(y_test, y_pred_proba[:, 1]) 
        auprc_val = average_precision_score(y_test, y_pred_proba[:, 1]) 
        brier_val = brier_score_loss(y_test, y_pred_proba[:, 1]) 
        
        # Extraer la prevalencia (tasa base) específica de la clase positiva (1)
        clases_temp, soportes_clases = np.unique(y_test, return_counts=True)
        indice_clase_1 = np.where(clases_temp == 1)[0][0]
        tasa_base = soportes_clases[indice_clase_1] / total_instancias

        # Imprimir resultados binarios
        print(f"F1-Score (Clase 1): {f1_clase1_val:.4f}") 
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") 
        print(f"AUPRC: {auprc_val:.4f}") 
        print(f"AUC-ROC: {auc_roc_val:.4f}") 
        print(f"Brier Score: {brier_val:.4f}") 

    # Analizar si el modelo ofrece una mejora significativa frente al modelo aleatorio (validación Lift)
    print("\n" + "-" * 60)
    print(f"Validación de Lift (en AUPRC): {target_name.upper()}")
    print("-" * 60)
    print(f"Total episodios de prueba: {total_instancias}")
    print(f"Tasa base (Prevalencia Azar): {tasa_base:.4f} ({tasa_base*100:.2f}%)")
    print(f"AUPRC Obtenido por tu modelo: {auprc_val:.4f}")
    
    # Calcular el nivel de lift real contra el modelo aleatorio base
    umbral_minimo = tasa_base * 3.0
    lift_real = auprc_val / tasa_base
    
    print(f"Lift real logrado: {lift_real:.2f}x")
    
    # Para datasets desbalanceados (<15% prevalencia), verificar si el AUPRC triplica el azar
    if tasa_base < 0.15: 
        print(f"AUPRC Mínimo exigido (Tasa Base x 3.0): {umbral_minimo:.4f}")
        if auprc_val > umbral_minimo:
            print("Resultado: Cumple condición de Lift > 3.0")
        else:
            print("Resultado: No cumple condición de Lift > 3.0")
    else:
        print("Resultado: Target suficientemente balanceado")

    # 7. Extraer Feature Importances (Importancia de variables del árbol)
    importancias = mejor_modelo.feature_importances_ 
    
    # Construir un dataframe cruzando los nombres de variables con sus respectivos pesos de importancia
    df_importancias = pd.DataFrame({ 
        'Variable': features, 
        'Importancia_Relativa': importancias 
    }).sort_values(by='Importancia_Relativa', ascending=False) 
    
    # Filtrar solo las variables que tienen un aporte mayor a 0 al modelo
    df_importancias = df_importancias[df_importancias['Importancia_Relativa'] > 0] 
    
    # Exportar el ranking de variables predictoras en formato CSV
    ruta_imp = os.path.join(dir_resultados, f"RF_Importancia_Predictores_{target_name}.csv") 
    df_importancias.to_csv(ruta_imp, index=False) 
    
    print(f"\nImportancias de Variables guardadas en: {ruta_imp}") 
    
    # Liberar la memoria final después de toda la ejecución
    del df_test_maestro, X_test, y_test 
    gc.collect() 
    print("="*60, "\n")

In [2]:
entrenar_evaluar_rf('MORTALIDAD')

Iniciando entrenamiento y evaluación de Random Forest para la variable objetivo: MORTALIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (780416, 110) | Clases: 2 (Multiclase: False)
[3/5] Configurando Grid Search CV (K=5) ...
[4/5] Entrenando modelo y evaluando configuraciones ...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV 1/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.599 total time=  49.1s
[CV 2/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.597 total time=  48.1s
[CV 3/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.598 total time=  47.7s
[CV 4/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.600 total time=  47.8s
[CV 5/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.599 total time=  48.0s
[CV 1/5] END max_depth=15, min_samples_split=10, n_estimators=200;, score=0.599 total time= 1.6min
[C

In [3]:
entrenar_evaluar_rf('SEVERIDAD')

Iniciando entrenamiento y evaluación de Random Forest para la variable objetivo: SEVERIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (780416, 110) | Clases: 4 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) ...
[4/5] Entrenando modelo y evaluando configuraciones ...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV 1/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.727 total time=  51.0s
[CV 2/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.726 total time=  49.4s
[CV 3/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.726 total time=  49.9s
[CV 4/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.728 total time=  49.8s
[CV 5/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.726 total time=  49.8s
[CV 1/5] END max_depth=15, min_samples_split=10, n_estimators=200;, score=0.727 total time= 1.6min
[CV 

In [4]:
entrenar_evaluar_rf('CONSUMO_RECURSOS')

Iniciando entrenamiento y evaluación de Random Forest para la variable objetivo: CONSUMO_RECURSOS
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (780416, 110) | Clases: 3 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) ...
[4/5] Entrenando modelo y evaluando configuraciones ...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV 1/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.717 total time=  45.9s
[CV 2/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.716 total time=  45.2s
[CV 3/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.717 total time=  45.1s
[CV 4/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.715 total time=  44.7s
[CV 5/5] END max_depth=15, min_samples_split=10, n_estimators=100;, score=0.715 total time=  43.8s
[CV 1/5] END max_depth=15, min_samples_split=10, n_estimators=200;, score=0.718 total time= 1.4m